In [7]:
import tkinter as tk
from tkinter import filedialog, messagebox, scrolledtext
import tkinter.font as tkFont
import os
import time

# Initialize an empty list for selected files
FILES = []

# Brute Force Search Algorithm
def brute_force_search(text, pattern, whole_word, case_sensitive):
    results = []
    if not case_sensitive:
        text = text.lower()
        pattern = pattern.lower()

    for i in range(len(text) - len(pattern) + 1):
        match = True
        for j in range(len(pattern)):
            if text[i + j] != pattern[j]:
                match = False
                break
        if match:
            if whole_word:
                if (i == 0 or not text[i - 1].isalnum()) and (i + len(pattern) == len(text) or not text[i + len(pattern)].isalnum()):
                    results.append(i)
            else:
                results.append(i)
    return results

# Knuth-Morris-Pratt (KMP) Search Algorithm
def kmp_search(text, pattern, whole_word, case_sensitive):
    results = []
    if not case_sensitive:
        text = text.lower()
        pattern = pattern.lower()

    # KMP Preprocessing: Build LPS array
    lps = [0] * len(pattern)
    length = 0
    i = 1
    while i < len(pattern):
        if pattern[i] == pattern[length]:
            length += 1
            lps[i] = length
            i += 1
        else:
            if length != 0:
                length = lps[length - 1]
            else:
                lps[i] = 0
                i += 1

    i = 0  # index for text
    j = 0  # index for pattern
    while i < len(text):
        if pattern[j] == text[i]:
            i += 1
            j += 1
        if j == len(pattern):
            if whole_word:
                if (i - j == 0 or not text[i - j - 1].isalnum()) and (i == len(text) or not text[i].isalnum()):
                    results.append(i - j)
            else:
                results.append(i - j)
            j = lps[j - 1]
        elif i < len(text) and pattern[j] != text[i]:
            if j != 0:
                j = lps[j - 1]
            else:
                i += 1
    return results

# GUI Setup
root = tk.Tk()
root.title("Word Search Application")
root.geometry("600x600")
root.configure(bg="black")  # Set background color to black

# Function to select files
def select_files():
    global FILES
    FILES = filedialog.askopenfilenames(title="Select Files to Search", filetypes=[("Text Files", "*.txt")])
    if FILES:
        messagebox.showinfo("Files Selected", f"{len(FILES)} files selected.")

# Font settings
font_setting = tkFont.Font(family="Calibri", size=12, weight="bold")  # Set to Calibri and bold

# Search term entry
tk.Label(root, text="Enter Search Term:", bg="black", fg="white", font=font_setting).pack(pady=5)
search_term_entry = tk.Entry(root, width=40, bg="#333333", fg="white", insertbackground="white", font=font_setting)
search_term_entry.pack(pady=5)

# Whole word match checkbox
whole_word_var = tk.BooleanVar()
whole_word_checkbox = tk.Checkbutton(root, text="Whole Word Match", variable=whole_word_var, bg="black", fg="white", selectcolor="black", font=font_setting)
whole_word_checkbox.pack()

# Case sensitivity checkbox
case_sensitive_var = tk.BooleanVar()
case_sensitive_checkbox = tk.Checkbutton(root, text="Case Sensitive", variable=case_sensitive_var, bg="black", fg="white", selectcolor="black", font=font_setting)
case_sensitive_checkbox.pack()

# Result display area
result_area = scrolledtext.ScrolledText(root, width=70, height=20, bg="#1e1e1e", fg="white", insertbackground="white", font=font_setting)
result_area.pack(pady=10)

# Search Function
def search():
    if not FILES:
        messagebox.showerror("Error", "Please select files to search.")
        return

    search_term = search_term_entry.get()
    if not search_term:
        messagebox.showerror("Error", "Please enter a search term.")
        return

    whole_word = whole_word_var.get()
    case_sensitive = case_sensitive_var.get()
    result_area.delete("1.0", tk.END)

    for file_path in FILES:
        try:
            with open(file_path, "r", encoding="utf-8") as file:
                lines = file.readlines()  # Read file line by line

            # Concatenate lines to form full text for searching
            text = ''.join(lines)

            # Brute Force Search
            start_time = time.time()
            brute_force_matches = brute_force_search(text, search_term, whole_word, case_sensitive)
            brute_force_time = time.time() - start_time

            # KMP Search
            start_time = time.time()
            kmp_matches = kmp_search(text, search_term, whole_word, case_sensitive)
            kmp_time = time.time() - start_time

            # Consolidate all positions and remove duplicates
            combined_positions = sorted(set(brute_force_matches + kmp_matches))

            # Display results
            result_area.insert(tk.END, f"\n\nFile: {os.path.basename(file_path)}\n")
            result_area.insert(tk.END, f"Brute Force Search Time: {brute_force_time:.6f} seconds\n")
            result_area.insert(tk.END, f"KMP Search Time: {kmp_time:.6f} seconds\n")

            if combined_positions:
                result_area.insert(tk.END, "\n")
                for pos in combined_positions:
                    line_num = next(i + 1 for i, line in enumerate(lines) if text.find(search_term, pos) < text.find(line) + len(line))
                    result_area.insert(tk.END, f"Position: {pos}, Line: {line_num}\n")
            else:
                result_area.insert(tk.END, "No occurrences found.\n")

        except UnicodeDecodeError as e:
            messagebox.showerror("Error", f"Could not open file {file_path} due to encoding issue: {e}")

# Select Files button
select_files_button = tk.Button(root, text="Select Files", command=select_files, bg="#444444", fg="white", activebackground="#555555", font=font_setting)
select_files_button.pack(pady=5)

# Search button
search_button = tk.Button(root, text="Search", command=search, bg="#444444", fg="white", activebackground="#555555", font=font_setting)
search_button.pack(pady=10)

root.mainloop()
